# RunPod SFT — Gemma 4 31B (train.sh)

Minimal RunPod notebook. One script runs the full pipeline:

**preflight → SFT + push LoRA → merge + push → build llama.cpp → GGUF + push**

## Pod setup

1. **A100 80GB** PyTorch template (seq 8192 4-bit needs ~80 GB VRAM).
2. Network volume mounted at `/workspace`.
3. Accept the [google/gemma-4-31B-it](https://huggingface.co/google/gemma-4-31B-it) license.
4. Write-capable `HF_TOKEN` for Hub pushes (adapter, merged, GGUF repos).

Artifacts land under `/workspace/gemma4-31b-manim-{ft,merged,gguf}`.

On **40GB** GPUs: `EPOCHS=1 SEQ_LEN=2048 bash apps/sft/train.sh`.

## 1. Secrets + GPU

In [ ]:
import os

# Required — Hugging Face write token (Gemma license + Hub pushes)
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "hf_...")

# Optional W&B
# os.environ["WANDB_API_KEY"] = "..."
# os.environ["WANDB_ENTITY"] = "your-entity"
# os.environ["WANDB_PROJECT_SFT"] = "aos-sft"

assert os.environ["HF_TOKEN"].startswith("hf_") and len(os.environ["HF_TOKEN"]) > 10, (
    "Set a real HF_TOKEN before training"
)
print("HF_TOKEN:", os.environ["HF_TOKEN"][:6] + "...")
print("WANDB:", "yes" if os.environ.get("WANDB_API_KEY", "").strip() else "no")

In [ ]:
!nvidia-smi

## 2. Clone + deps

Edit `REPO_URL` if you fork.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/nabin2004/AOS.git"
REPO_DIR = Path("/workspace/AOS")

if REPO_DIR.is_dir() and (REPO_DIR / ".git").is_dir():
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

!pip install -q uv
!uv sync --package sft

## 3. Train end-to-end

Runs [`apps/sft/train.sh`](train.sh): preflight, SFT, adapter Hub push, merge, GGUF export, GGUF Hub push.

Override epochs / seq length via env: `EPOCHS=3` or `SEQ_LEN=2048`.

In [ ]:
# Full pipeline (default 1 epoch). Examples:
#   !EPOCHS=3 bash apps/sft/train.sh
#   !SEQ_LEN=2048 bash apps/sft/train.sh
!bash apps/sft/train.sh

## 4. Optional — infer

Uses the local LoRA adapter under `/workspace`.

In [ ]:
!uv run --package sft python apps/sft/infer.py \
  --adapter-dir /workspace/gemma4-31b-manim-ft \
  --runpod \
  --no-tools \
  --prompt "Create a short Manim scene explaining eigenvectors in 2D."